# Apache Spark com Delta Lake

Demonstração de operações CRUD (INSERT, UPDATE, DELETE) com **PySpark** e **Delta Lake**.

**Cenário:** Sistema de Gestão de Vendas — TechStore  
**Tabelas:** clientes, produtos, pedidos

In [1]:
import subprocess, os

# Limpa warehouse anterior usando rm -rf (confiavel no WSL/OneDrive)
for d in ['/tmp/spark-wh-delta', os.path.expanduser('~/metastore_db')]:
    r = subprocess.run(['rm', '-rf', d], capture_output=True, text=True)
    status = 'removido' if r.returncode == 0 else f'erro: {r.stderr.strip()}'
    print(f'{d} — {status}')


/tmp/spark-wh-delta — removido
/home/thiago/metastore_db — removido


In [2]:
from pyspark.sql import SparkSession
from delta import *
import logging

logging.getLogger('py4j').setLevel(logging.WARNING)


In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .config('spark.sql.warehouse.dir', '/tmp/spark-wh-delta')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0')
    .config('spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark


26/05/07 00:30:35 WARN Utils: Your hostname, LAPTOP-NGVDQKNB resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/07 00:30:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/mazuc/OneDrive/%c3%81rea%20de%20Trabalho/SATC/5%c2%b0%20FASE/Engenharia%20de%20Dados/Trabalho%20Apache%20spark%20iceberg%20e%20Data%20Lake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thiago/.ivy2/cache
The jars for the packages stored in: /home/thiago/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-45d4745d-1980-4a59-9032-8b9dfcc8ada0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 422ms :: artifacts dl 14ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

## Cenário — TechStore

Sistema de gestão de vendas de uma loja de eletrônicos com três entidades.

### Modelo ER
```
CLIENTES (1) ----< PEDIDOS >---- (N) PRODUTOS
```

- Um cliente pode realizar vários pedidos
- Um produto pode estar em vários pedidos

## DDL — Criação das Tabelas Delta

In [4]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS clientes (
        id       INT,
        nome     STRING,
        email    STRING,
        cidade   STRING,
        estado   STRING
    ) USING delta
""")
spark.sql("SELECT * FROM clientes").show()


26/05/07 00:30:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---+----+-----+------+------+
| id|nome|email|cidade|estado|
+---+----+-----+------+------+
+---+----+-----+------+------+



In [5]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS produtos (
        id        INT,
        nome      STRING,
        categoria STRING,
        preco     FLOAT,
        estoque   INT
    ) USING delta
""")
spark.sql("SELECT * FROM produtos").show()


[Stage 14:==========================================>             (38 + 8) / 50]

+---+----+---------+-----+-------+
| id|nome|categoria|preco|estoque|
+---+----+---------+-----+-------+
+---+----+---------+-----+-------+



In [6]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS pedidos (
        id           INT,
        cliente_id   INT,
        produto_id   INT,
        quantidade   INT,
        data_pedido  STRING,
        status       STRING
    ) USING delta
""")
spark.sql("SELECT * FROM pedidos").show()


+---+----------+----------+----------+-----------+------+
| id|cliente_id|produto_id|quantidade|data_pedido|status|
+---+----------+----------+----------+-----------+------+
+---+----------+----------+----------+-----------+------+



## INSERT — Inserindo Dados

In [7]:
spark.sql("""
    INSERT INTO clientes VALUES
        (1, 'Ana Silva',       'ana@email.com',      'Sao Paulo',      'SP'),
        (2, 'Carlos Oliveira', 'carlos@email.com',   'Rio de Janeiro', 'RJ'),
        (3, 'Maria Santos',    'maria@email.com',    'Curitiba',       'PR'),
        (4, 'Joao Costa',      'joao@email.com',     'Porto Alegre',   'RS'),
        (5, 'Fernanda Lima',   'fernanda@email.com', 'Belo Horizonte', 'MG')
""")
spark.sql("SELECT * FROM clientes").show()


+---+---------------+------------------+--------------+------+
| id|           nome|             email|        cidade|estado|
+---+---------------+------------------+--------------+------+
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|
+---+---------------+------------------+--------------+------+



In [8]:
spark.sql("""
    INSERT INTO produtos VALUES
        (1, 'Notebook Dell',      'Informatica',  3599.99, 15),
        (2, 'Smartphone Samsung', 'Celulares',    1299.00, 50),
        (3, 'Monitor LG 27',      'Informatica',   899.90, 30),
        (4, 'Teclado Mecanico',   'Perifericos',   349.90, 100),
        (5, 'Mouse Logitech',     'Perifericos',   159.90, 80)
""")
spark.sql("SELECT * FROM produtos").show()


[Stage 45:========================================>               (36 + 8) / 50]

+---+------------------+-----------+-------+-------+
| id|              nome|  categoria|  preco|estoque|
+---+------------------+-----------+-------+-------+
|  2|Smartphone Samsung|  Celulares| 1299.0|     50|
|  4|  Teclado Mecanico|Perifericos|  349.9|    100|
|  5|    Mouse Logitech|Perifericos|  159.9|     80|
|  1|     Notebook Dell|Informatica|3599.99|     15|
|  3|     Monitor LG 27|Informatica|  899.9|     30|
+---+------------------+-----------+-------+-------+



In [9]:
spark.sql("""
    INSERT INTO pedidos VALUES
        (1, 1, 2, 2, '2024-01-10', 'entregue'),
        (2, 2, 1, 1, '2024-01-12', 'entregue'),
        (3, 3, 3, 1, '2024-01-15', 'em_transporte'),
        (4, 1, 4, 1, '2024-01-20', 'processando'),
        (5, 4, 5, 3, '2024-01-22', 'cancelado')
""")
spark.sql("SELECT * FROM pedidos").show()


+---+----------+----------+----------+-----------+-------------+
| id|cliente_id|produto_id|quantidade|data_pedido|       status|
+---+----------+----------+----------+-----------+-------------+
|  3|         3|         3|         1| 2024-01-15|em_transporte|
|  4|         1|         4|         1| 2024-01-20|  processando|
|  5|         4|         5|         3| 2024-01-22|    cancelado|
|  2|         2|         1|         1| 2024-01-12|     entregue|
|  1|         1|         2|         2| 2024-01-10|     entregue|
+---+----------+----------+----------+-----------+-------------+



## Consulta com JOIN

In [10]:
spark.sql("""
    SELECT
        p.id         AS pedido_id,
        c.nome       AS cliente,
        pr.nome      AS produto,
        p.quantidade,
        p.status,
        p.data_pedido
    FROM pedidos p
    JOIN clientes c  ON p.cliente_id = c.id
    JOIN produtos pr ON p.produto_id = pr.id
    ORDER BY p.id
""").show(truncate=False)


+---------+---------------+------------------+----------+-------------+-----------+
|pedido_id|cliente        |produto           |quantidade|status       |data_pedido|
+---------+---------------+------------------+----------+-------------+-----------+
|1        |Ana Silva      |Smartphone Samsung|2         |entregue     |2024-01-10 |
|2        |Carlos Oliveira|Notebook Dell     |1         |entregue     |2024-01-12 |
|3        |Maria Santos   |Monitor LG 27     |1         |em_transporte|2024-01-15 |
|4        |Ana Silva      |Teclado Mecanico  |1         |processando  |2024-01-20 |
|5        |Joao Costa     |Mouse Logitech    |3         |cancelado    |2024-01-22 |
+---------+---------------+------------------+----------+-------------+-----------+



## UPDATE — Atualizando Dados

In [11]:
# Atualiza status do pedido 3 para entregue
spark.sql("UPDATE pedidos SET status = 'entregue' WHERE id = 3")
spark.sql("SELECT * FROM pedidos WHERE id = 3").show()


[Stage 81:=================================>                      (30 + 8) / 50]

+---+----------+----------+----------+-----------+--------+
| id|cliente_id|produto_id|quantidade|data_pedido|  status|
+---+----------+----------+----------+-----------+--------+
|  3|         3|         3|         1| 2024-01-15|entregue|
+---+----------+----------+----------+-----------+--------+



In [12]:
# Ajusta preco e estoque do produto 2
spark.sql("UPDATE produtos SET preco = 1199.00, estoque = 45 WHERE id = 2")
spark.sql("SELECT * FROM produtos WHERE id = 2").show()


+---+------------------+---------+------+-------+
| id|              nome|categoria| preco|estoque|
+---+------------------+---------+------+-------+
|  2|Smartphone Samsung|Celulares|1199.0|     45|
+---+------------------+---------+------+-------+



## DELETE — Removendo Dados

In [13]:
# Remove pedidos cancelados
spark.sql("DELETE FROM pedidos WHERE status = 'cancelado'")
spark.sql("SELECT * FROM pedidos").show()


+---+----------+----------+----------+-----------+-----------+
| id|cliente_id|produto_id|quantidade|data_pedido|     status|
+---+----------+----------+----------+-----------+-----------+
|  4|         1|         4|         1| 2024-01-20|processando|
|  2|         2|         1|         1| 2024-01-12|   entregue|
|  1|         1|         2|         2| 2024-01-10|   entregue|
|  3|         3|         3|         1| 2024-01-15|   entregue|
+---+----------+----------+----------+-----------+-----------+



## ALTER TABLE — Evolução de Schema

O Delta Lake suporta adicionar colunas sem recriar a tabela.

In [14]:
spark.sql("ALTER TABLE clientes ADD COLUMN telefone STRING")
spark.sql("SELECT * FROM clientes").show()


+---+---------------+------------------+--------------+------+--------+
| id|           nome|             email|        cidade|estado|telefone|
+---+---------------+------------------+--------------+------+--------+
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|    NULL|
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|    NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|    NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|    NULL|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|    NULL|
+---+---------------+------------------+--------------+------+--------+



In [15]:
spark.sql("UPDATE clientes SET telefone = '(11) 91234-5678' WHERE id = 1")
spark.sql("UPDATE clientes SET telefone = '(21) 99876-5432' WHERE id = 2")
spark.sql("SELECT * FROM clientes").show()


+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
+---+---------------+------------------+--------------+------+---------------+



## MERGE — Upsert (Insert or Update)

Insere o registro se não existir, ou atualiza se já existir.

In [16]:
spark.sql("""
    MERGE INTO clientes AS target
    USING (
        SELECT 6 AS id, 'Pedro Alves' AS nome, 'pedro@email.com' AS email,
               'Fortaleza' AS cidade, 'CE' AS estado, '(85) 98765-4321' AS telefone
    ) AS source
    ON target.id = source.id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
spark.sql("SELECT * FROM clientes").show()


+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  6|    Pedro Alves|   pedro@email.com|     Fortaleza|    CE|(85) 98765-4321|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
+---+---------------+------------------+--------------+------+---------------+



## Time Travel — Viagem no Tempo

O Delta Lake mantém um **transaction log** que permite consultar versões anteriores.

In [17]:
spark.sql("DESCRIBE HISTORY clientes").show(truncate=False)


+-------+-----------------------+------+--------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp 

In [18]:
from delta.tables import DeltaTable

tabela_path = '/tmp/spark-wh-delta/clientes'
print(DeltaTable.isDeltaTable(spark, tabela_path))

# Le versao 0 — estado logo apos o primeiro INSERT
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(tabela_path)
print('Clientes na versao 0 (so INSERT):')
df_v0.show()


True
Clientes na versao 0 (so INSERT):


+---+----+-----+------+------+
| id|nome|email|cidade|estado|
+---+----+-----+------+------+
+---+----+-----+------+------+



In [19]:
print('Clientes na versao atual:')
spark.sql("SELECT * FROM clientes").show()


Clientes na versao atual:
+---+---------------+------------------+--------------+------+---------------+
| id|           nome|             email|        cidade|estado|       telefone|
+---+---------------+------------------+--------------+------+---------------+
|  2|Carlos Oliveira|  carlos@email.com|Rio de Janeiro|    RJ|(21) 99876-5432|
|  6|    Pedro Alves|   pedro@email.com|     Fortaleza|    CE|(85) 98765-4321|
|  1|      Ana Silva|     ana@email.com|     Sao Paulo|    SP|(11) 91234-5678|
|  5|  Fernanda Lima|fernanda@email.com|Belo Horizonte|    MG|           NULL|
|  4|     Joao Costa|    joao@email.com|  Porto Alegre|    RS|           NULL|
|  3|   Maria Santos|   maria@email.com|      Curitiba|    PR|           NULL|
+---+---------------+------------------+--------------+------+---------------+



In [20]:
spark.stop()
print('Sessao Spark encerrada.')


Sessao Spark encerrada.
